In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the 20 newsgroups dataset
newsgroups_data = fetch_20newsgroups(categories=['comp.graphics'], remove=('headers', 'footers', 'quotes'))

# Initialize TfidfVectorizer
# You can customize parameters like stop_words, ngram_range, max_df, min_df
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5)

# Fit and transform the data
tfidf_matrix = vectorizer.fit_transform(newsgroups_data.data)

print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")
print(f"Number of features (unique terms): {len(vectorizer.get_feature_names_out())}")

In [ ]:
import numpy as np
import numpy.linalg as la

In [ ]:
A = tfidf_matrix

In [ ]:
A.shape

In [ ]:
D1 = np.sum(A, axis=1)
D2 = np.sum(A.T, axis=1)
# Calculate D1^(-1/2)
D1 = np.squeeze(np.asarray(D1))
D2 = np.squeeze(np.asarray(D2))
D1_inv_sqrt = np.diag(1.0 / np.sqrt(D1))
D2_inv_sqrt = np.diag(1.0 / np.sqrt(D2))


In [ ]:
An = D1_inv_sqrt@A@D2_inv_sqrt

In [ ]:
U, s, Vt = np.linalg.svd(An)

In [ ]:
idx_sv = s.argsort()
sorted_singular_values = s[idx_sv]

In [ ]:

# 2. Set a threshold
threshold = 0.01

# 3. Create a boolean mask
mask = sorted_singular_values < threshold
true_count = np.count_nonzero(mask)
sorted_singular_values[:true_count]=0

In [ ]:
import pandas as pd
EVAL_LIMIT = 10
eig_val_idx = [(i+1) for i in range(2*EVAL_LIMIT)]
spec_gap = {"eig_val_index": eig_val_idx, "eig_vals": sorted_singular_values[:2*EVAL_LIMIT]}
df_sg = pd.DataFrame.from_dict(spec_gap, orient="columns")
df_sg["gap"] = df_sg["eig_vals"].diff()

In [ ]:
sel_gap = df_sg.gap ==  df_sg.gap.max()
df_sg[sel_gap]

In [ ]:
from math import ceil, log
k = 19 # from spectral gap analysis
l = ceil(log(k,2))

In [ ]:
Ul = D1_inv_sqrt @ U
Vl = D2_inv_sqrt @ Vt

In [ ]:
Ul = Ul[:, 1:(l+1)]
Vl = Vl[:, 1:(l+1)]

In [ ]:
Z = np.vstack((Ul, Vl))

In [ ]:
df_Z = pd.DataFrame(Z)
df_Z.columns = ["emb-" + str(i+1) for i in range(l)]
rows_with_nan_mask = df_Z.isna().any(axis=1)
index_of_rows_with_nan = df_Z[rows_with_nan_mask].index
df_Z.replace([np.inf, -np.inf], 0, inplace=True)
df_Z = df_Z.fillna(0)

In [ ]:
index_of_rows_with_nan

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=k, random_state=0, n_init="auto").fit(df_Z)

In [ ]:
df_Z["cluster"] = kmeans.labels_

In [ ]:
df_Z.cluster.value_counts().shape

In [ ]:
df_words = df_Z.loc[Ul.shape[0]:, :]

In [ ]:
df_words.cluster.value_counts()

In [ ]:
words = vectorizer.get_feature_names_out()

In [ ]:
cluster_sel = df_words.cluster == 17

In [ ]:
words[cluster_sel]